In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split

#  Load data 
df = pd.read_csv('customer.csv')

# keep only relevant columns
df = df.iloc[:, 2:]

print("Shape:", df.shape)
print("\nSample data:")
print(df.head(10))
print("\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].unique()}")

Shape: (50, 3)

Sample data:
    review education purchased
0  Average    School        No
1     Poor        UG        No
2     Good        PG        No
3     Good        PG        No
4  Average        UG        No
5  Average    School       Yes
6     Good    School        No
7     Poor    School       Yes
8  Average        UG        No
9     Good        UG       Yes

Unique values per column:
  review: <ArrowStringArray>
['Average', 'Poor', 'Good']
Length: 3, dtype: str
  education: <ArrowStringArray>
['School', 'UG', 'PG']
Length: 3, dtype: str
  purchased: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [3]:
#  Split BEFORE encoding 
# always split first, encode after
# reason: prevents data leakage (same as scaling)

X = df.drop('purchased', axis=1)   # features → review, education
y = df['purchased']                 # target   → Yes/No

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("\nX_train sample:")
print(X_train.head(8))
print("\ny_train sample:")
print(y_train.head(8).values)

X_train shape: (40, 2)
X_test shape : (10, 2)

X_train sample:
     review education
12     Poor    School
4   Average        UG
37  Average        PG
8   Average        UG
3      Good        PG
6      Good    School
41     Good        PG
46     Poor        PG

y_train sample:
<ArrowStringArray>
['No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No']
Length: 8, dtype: str


In [4]:
#  ORDINAL ENCODER 

# STEP 1: define the correct order for each column
# ORDER MATTERS — you must specify it manually
# wrong order = wrong relationships learned by model

oe = OrdinalEncoder(categories=[
    ['Poor', 'Average', 'Good'],   # review:    0, 1, 2
    ['School', 'UG', 'PG']        # education: 0, 1, 2
])

# STEP 2: fit on train only → learns the mapping
oe.fit(X_train)

# see what mapping it learned
print("Categories learned:")
print(f"  review    : {oe.categories_[0]}")
print(f"  education : {oe.categories_[1]}")
print()

# STEP 3: transform train and test
X_train_encoded = oe.transform(X_train)
X_test_encoded  = oe.transform(X_test)

# STEP 4: convert back to dataframe for readability
X_train_encoded = pd.DataFrame(X_train_encoded, columns=X_train.columns)
X_test_encoded  = pd.DataFrame(X_test_encoded,  columns=X_test.columns)

#  Compare before and after 
print("BEFORE encoding:")
print(X_train.head(8).to_string())

print("\nAFTER encoding:")
print(X_train_encoded.head(8).to_string())

Categories learned:
  review    : ['Poor' 'Average' 'Good']
  education : ['School' 'UG' 'PG']

BEFORE encoding:
     review education
12     Poor    School
4   Average        UG
37  Average        PG
8   Average        UG
3      Good        PG
6      Good    School
41     Good        PG
46     Poor        PG

AFTER encoding:
   review  education
0     0.0        0.0
1     1.0        1.0
2     1.0        2.0
3     1.0        1.0
4     2.0        2.0
5     2.0        0.0
6     2.0        2.0
7     0.0        2.0


In [6]:
# manually verify encoding is correct
print("Encoding verification:")

mapping_review    = {'Poor':0, 'Average':1, 'Good':2}
mapping_education = {'School':0, 'UG':1, 'PG':2}

for original, encoded in zip(
    X_train['review'].values[:6],
    X_train_encoded['review'].values[:6]
):
    expected = mapping_review[original]
    status   = "Right" if encoded == expected else "Wrong"
    print(f"  {original:<8} → {int(encoded)} {status}")

print()
for original, encoded in zip(
    X_train['education'].values[:6],
    X_train_encoded['education'].values[:6]
):
    expected = mapping_education[original]
    status   = "Right" if encoded == expected else "Wrong"
    print(f"  {original:<8} → {int(encoded)} {status}")

Encoding verification:
  Poor     → 0 Right
  Average  → 1 Right
  Average  → 1 Right
  Average  → 1 Right
  Good     → 2 Right
  Good     → 2 Right

  School   → 0 Right
  UG       → 1 Right
  PG       → 2 Right
  UG       → 1 Right
  PG       → 2 Right
  School   → 0 Right


In [7]:
#  LabelEncoder 

# STEP 1: create encoder
le = LabelEncoder()

# STEP 2: fit on y_train → learns classes
le.fit(y_train)

# see what it learned
print("Classes learned:")
print(f"  {le.classes_}")
print(f"  No  → {le.transform(['No'])[0]}")   # No  = 0
print(f"  Yes → {le.transform(['Yes'])[0]}")  # Yes = 1

# STEP 3: transform train AND test with same encoder
y_train_encoded = le.transform(y_train)
y_test_encoded  = le.transform(y_test)

print("\nBefore encoding (text):")
print(list(y_train[:10]))

print("\nAfter encoding (numbers):")
print(y_train_encoded[:10])

print("\nClass distribution:")
print(f"  0 (No) : {(y_train_encoded==0).sum()}")
print(f"  1 (Yes): {(y_train_encoded==1).sum()}")

Classes learned:
  ['No' 'Yes']
  No  → 0
  Yes → 1

Before encoding (text):
['No', 'No', 'Yes', 'No', 'No', 'No', 'Yes', 'No', 'Yes', 'No']

After encoding (numbers):
[0 0 1 0 0 0 1 0 1 0]

Class distribution:
  0 (No) : 21
  1 (Yes): 19


In [8]:
#  danger of wrong order 

# WITHOUT defining order → sklearn uses alphabetical
oe_wrong = OrdinalEncoder()   # no categories defined
oe_wrong.fit(X_train)

print("WITHOUT defining order (alphabetical — WRONG):")
print(f"  review    : {oe_wrong.categories_[0]}")
print(f"  education : {oe_wrong.categories_[1]}")

print("\nWITH correct order (YOUR defined order):")
print(f"  review    : {oe.categories_[0]}")
print(f"  education : {oe.categories_[1]}")

WITHOUT defining order (alphabetical — WRONG):
  review    : ['Average' 'Good' 'Poor']
  education : ['PG' 'School' 'UG']

WITH correct order (YOUR defined order):
  review    : ['Poor' 'Average' 'Good']
  education : ['School' 'UG' 'PG']


In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

#  encode features 
oe = OrdinalEncoder(categories=[
    ['Poor', 'Average', 'Good'],
    ['School', 'UG', 'PG']
])
X_train_enc = oe.fit_transform(X_train)
X_test_enc  = oe.transform(X_test)

#  encode target 
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

#  train model 
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train_enc, y_train_enc)

#  predict and evaluate 
y_pred = model.predict(X_test_enc)
print(f"Accuracy: {accuracy_score(y_test_enc, y_pred):.4f}")

#  decode predictions back to Yes/No 
y_pred_labels = le.inverse_transform(y_pred)
print("\nPredictions (decoded):", y_pred_labels[:10])
print("Actual     (decoded) :", list(y_test[:10]))

Accuracy: 0.6000

Predictions (decoded): ['No' 'No' 'No' 'No' 'No' 'Yes' 'No' 'No' 'No' 'No']
Actual     (decoded) : ['No', 'No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'No', 'Yes', 'Yes']
